# Best Simulation Summary by Cell Line

This notebook imports the run-level comparison results for Campaigns 1–6 and creates a table showing the best simulation on a **Wet Cell Mass (WCM)** basis for:

- Space-Time Yield (STY)
- Feed Conversion Efficiency (FCE)
- Protein Conversion Efficiency (PCE)
- Medium Conversion Efficiency (MCE)

Each campaign represents a cell line, so the first column is named **Cell line**.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display


# Resolve paths from the workspace layout. Override ROOT manually if needed.
ROOT = Path.cwd().parent
if not (ROOT / "Campaign_1_fixed_stoichiometry_Humbird").exists():
    ROOT = Path(r"C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code")

CAMPAIGN_FILES = {
    "1": ROOT / "Campaign_1_fixed_stoichiometry_Humbird" / "06_Analysis" / "summary_comparison.csv",
    "2": ROOT / "Campaign_6_fixed_stoichiometry_Humbird_enhanced" / "06_Analysis" / "summary_comparison.csv",
    "3": ROOT / "Campaign_2_fixed_stoichiometry_GFI" / "06_Analysis" / "summary_comparison.csv",
    # "3": ROOT / "Campaign_3_dynamic_stoichiometry_GFI" / "06_Analysis" / "summary_comparison.csv",
    "4": ROOT / "Campaign_4_mode_comparison_excesskinetics" / "06_Analysis" / "summary_comparison.csv",
    "5": ROOT / "Campaign_5_mode_comparison_adjustedexcesskinetics" / "06_Analysis" / "summary_comparison.csv",
}

# (Model, Cell Properties) metadata per cell line
CAMPAIGN_INFO = {
    "1": ("Fixed", "Humbird (2020) Wildtype"),
    "2": ("Fixed", "Humbird (2020) Enhanced"),
    "3": ("Fixed", "Frohlich et al. (2025)"),
    # "3": ("Growth rate dependent", "Frohlich et al. (2025)"),
    "4": ("Substrate Dependent", "Frohlich et al. (2025)"),
    "5": ("Substrate Dependent Enhanced", "Frohlich et al. (2025)"),
}

# Metric columns (on a wet cell mass basis where applicable)
STY_COLUMN = "STY_WCM_kg_m3_day"
FCE_COLUMN = "feed_conversion_efficiency_kg_kg_WCM"
PCE_COLUMN = "protein_conversion_efficiency_kg_kg"
MCE_COLUMN = "medium_conversion_efficiency_m3_kg_WCM"
REQUIRED_COLUMNS = {"mode", STY_COLUMN, FCE_COLUMN, PCE_COLUMN, MCE_COLUMN}

MODE_ABBREVIATIONS = {
    "Batch": "Batch",
    "Batch (startup Fedbatch)": "Batch",
    "Fedbatch": "FB",
    "Concentrated Fedbatch": "CF",
    "Concentrated FedBatch": "CF",
    "Continuous Perfusion": "CP",
    "Chemostat": "Chem",
}

STRATEGY_ABBREVIATIONS = {
    "constant": "Constant",
    "csfr": "CSFR",
    "exponential": "Exponential",
    "linear": "Linear",
    "gsp": "GSP",
    "cspr": "CSPR",
    "tbs": "TBS",
    "tbp": "TBP",
    "toxicity_based_step": "TBS",
    "toxicity_based_proportional": "TBP",
}


def format_simulation_name(row: pd.Series) -> str:
    """Return a concise mode and strategy name from the source row."""
    if row is None:
        return "No data"

    simulation_type = str(row.get("simulation_type", row.get("mode", ""))).strip()
    run_id = str(row.get("run_id", row.get("run", ""))).strip()
    parent_name = str(row.get("parent_name", ""))
    full_id = f"{parent_name}_{run_id}".casefold() if parent_name else run_id.casefold()

    if simulation_type.casefold().startswith("batch") or full_id.startswith("batch_") or "batch (startup fedbatch)" in full_id:
        return "Batch"
    if simulation_type.casefold().startswith("chemostat") or full_id.startswith("chemostat_"):
        return "Chem"

    mode_abbr = MODE_ABBREVIATIONS.get(simulation_type, simulation_type)
    for token, label in sorted(STRATEGY_ABBREVIATIONS.items(), key=lambda item: len(item[0]), reverse=True):
        if re.search(rf"(?:^|_){re.escape(token)}(?:_|$)", full_id):
            return f"{mode_abbr} ({label})"

    return mode_abbr or "Unknown"


def load_best_simulation_table() -> pd.DataFrame:
    rows = []
    kpis = (
        ("STY", STY_COLUMN, "max", "kg m⁻³ d⁻¹"),
        ("FCE", FCE_COLUMN, "min", "kg kg⁻¹"),
        ("PCE", PCE_COLUMN, "min", "kg kg⁻¹"),
        ("MCE", MCE_COLUMN, "min", "m³ kg⁻¹"),
    )

    for cell_line, csv_path in CAMPAIGN_FILES.items():
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing input for Cell line {cell_line}: {csv_path}")

        data = pd.read_csv(csv_path)
        missing = REQUIRED_COLUMNS.difference(data.columns)
        if missing:
            raise ValueError(f"Cell line {cell_line} is missing required columns: {sorted(missing)}")

        data = data.copy()
        for _, column, _, _ in kpis:
            data[column] = pd.to_numeric(data[column], errors="coerce")
        data = data.replace([float("inf"), float("-inf")], pd.NA)

        best_rows = {
            label: (
                data.loc[data[column].idxmax()]
                if direction == "max" and data[column].notna().any()
                else data.loc[data[column].idxmin()]
                if direction == "min" and data[column].notna().any()
                else None
            )
            for label, column, direction, _ in kpis
        }

        model_name, cell_properties = CAMPAIGN_INFO[cell_line]
        row = {
            "Cell line": cell_line,
            "Model": model_name,
            "Cell Properties": cell_properties,
        }
        for label, column, _, unit in kpis:
            best_row = best_rows[label]
            row[f"{label} — Best simulation"] = format_simulation_name(best_row)
            row[f"{label} — Value ({unit})"] = best_row[column] if best_row is not None else pd.NA
        rows.append(row)

        best_summary = ", ".join(
            f"best {label} = {format_simulation_name(best_rows[label])}" for label, _, _, _ in kpis
        )
        print(f"Cell line {cell_line}: {len(data)} runs loaded; {best_summary}")

    return pd.DataFrame(rows)


best_simulation_table = load_best_simulation_table()

# Paper-style formatting: black text, a restrained monochrome palette, clear borders,
# readable spacing, and no distracting notebook index column.
value_columns = [column for column in best_simulation_table.columns if " — Value" in column]
strategy_columns = [column for column in best_simulation_table.columns if " — Best simulation" in column]
paper_table = (
    best_simulation_table.style
    .format(
        {
            "STY — Value (kg m⁻³ d⁻¹)": "{:.2f}",
            "FCE — Value (kg kg⁻¹)": "{:.2f}",
            "PCE — Value (kg kg⁻¹)": "{:.2f}",
            "MCE — Value (m³ kg⁻¹)": "{:.2f}",
        },
        na_rep="",
    )
    .set_properties(
        **{
            "font-family": "Georgia, 'Times New Roman', serif",
            "font-size": "10pt",
            "color": "#000000",
            "background-color": "#ffffff",
            "border": "1px solid #7f7f7f",
            "padding": "7px 9px",
            "vertical-align": "middle",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("border-top", "1.5px solid #000000"),
                    ("border-bottom", "1.5px solid #000000"),
                    ("width", "100%"),
                    ("color", "#000000"),
                    ("background-color", "#ffffff"),
                ],
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("text-align", "left"),
                    ("font-family", "Georgia, 'Times New Roman', serif"),
                    ("font-size", "10pt"),
                    ("font-weight", "bold"),
                    ("color", "#000000"),
                    ("padding-bottom", "6px"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f2f2f2"),
                    ("color", "#000000"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border-bottom", "1px solid #000000"),
                    ("white-space", "normal"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("color", "#000000"),
                    ("background-color", "#ffffff"),
                ],
            },
            {
                "selector": "tbody tr:nth-child(even) td",
                "props": [("background-color", "#fafafa")],
            },
            {
                "selector": "tbody td:nth-child(1)",
                "props": [("text-align", "center"), ("font-weight", "bold")],
            },
            {
                "selector": ", ".join(
                    f"tbody td:nth-child({best_simulation_table.columns.get_loc(column) + 1})"
                    for column in strategy_columns
                ),
                "props": [("text-align", "left")],
            },
            {
                "selector": ", ".join(
                    f"tbody td:nth-child({best_simulation_table.columns.get_loc(column) + 1})"
                    for column in value_columns
                ),
                "props": [("text-align", "right"), ("font-variant-numeric", "tabular-nums")],
            },
        ]
    )
    .hide(axis="index")
)

paper_table = paper_table.set_caption("Best simulation results by cell line (WCM basis)")
display(paper_table)

Cell line 1: 17 runs loaded; best STY = CP (Constant), best FCE = Chem, best PCE = Chem, best MCE = FB (CSFR)
Cell line 2: 17 runs loaded; best STY = CP (CSPR), best FCE = FB (Constant), best PCE = CP (GSP), best MCE = CP (Constant)
Cell line 3: 17 runs loaded; best STY = CP (TBP), best FCE = Batch, best PCE = Batch, best MCE = FB (CSFR)
Cell line 4: 17 runs loaded; best STY = CP (CSPR), best FCE = Batch, best PCE = CF (GSP), best MCE = Batch
Cell line 5: 17 runs loaded; best STY = CP (TBP), best FCE = Chem, best PCE = Batch, best MCE = Batch


Cell line,Model,Cell Properties,STY — Best simulation,STY — Value (kg m⁻³ d⁻¹),FCE — Best simulation,FCE — Value (kg kg⁻¹),PCE — Best simulation,PCE — Value (kg kg⁻¹),MCE — Best simulation,MCE — Value (m³ kg⁻¹)
1,Fixed,Humbird (2020) Wildtype,CP (Constant),18.79,Chem,1.64,Chem,2.50,FB (CSFR),0.11
2,Fixed,Humbird (2020) Enhanced,CP (CSPR),74.68,FB (Constant),0.67,CP (GSP),1.33,CP (Constant),0.01
3,Fixed,Frohlich et al. (2025),CP (TBP),39.37,Batch,0.64,Batch,1.96,FB (CSFR),0.04
4,Substrate Dependent,Frohlich et al. (2025),CP (CSPR),8.36,Batch,1.45,CF (GSP),6.93,Batch,0.25
5,Substrate Dependent Enhanced,Frohlich et al. (2025),CP (TBP),42.56,Chem,0.69,Batch,1.64,Batch,0.04
